In [0]:
# LangGraph + OpenAI Agent Setup
%pip install langgraph langchain-openai mlflow openai
dbutils.library.restartPython()

%pip install langchain
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

# LangGraph + OpenAI Agent Setup
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import mlflow

# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = "api-key"

# Connect to OpenAI
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.1
)

print(" OpenAI LLM connected!")

# Quick test
response = llm.invoke("Say hello in one sentence!")
print(f" Response: {response.content}")

 OpenAI LLM connected!
 Response: Hello, how are you doing today?


In [0]:
# Tools for LangGraph Agent

from langchain_core.tools import tool

@tool
def get_sales_data(question: str) -> str:
    """Use this to get sales data from bronze_sales_data delta table"""
    df = spark.sql("SELECT * FROM bronze_sales_data")
    return f"Sales Data:\n{df.toPandas().to_string()}"

@tool
def get_api_users(question: str) -> str:
    """Use this to get user data from bronze_api_users delta table"""
    df = spark.sql("SELECT * FROM bronze_api_users")
    return f"Users Data:\n{df.toPandas().to_string()}"

@tool
def get_audit_log(question: str) -> str:
    """Use this to get ingestion audit log from bronze_audit_log delta table"""
    df = spark.sql("SELECT * FROM bronze_audit_log")
    return f"Audit Log:\n{df.toPandas().to_string()}"

print("Tools created!")
print("get_sales_data")
print("get_api_users")
print("get_audit_log")

Tools created!
get_sales_data
get_api_users
get_audit_log


In [0]:
# LangGraph Agent + MLflow LLMOps
import mlflow
import time

# Enable MLflow tracking
mlflow.set_experiment("/Users/theepankumar72@gmail.com/langgraph_agent_llmops")
mlflow.langchain.autolog()

# Create LangGraph agent
tools = [get_sales_data, get_api_users, get_audit_log]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful data analyst. You have access to Bronze Delta tables with sales, users and audit data. Always use tools to fetch data before answering."
)

print("LangGraph + OpenAI Agent created!")

# Test agent with MLflow tracking
@mlflow.trace
def ask_langgraph_agent(question):
    print(f"\n Question: {question}") 
    start_time = time.time()
    
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    duration = time.time() - start_time
    answer = response["messages"][-1].content
    
    print(f" Answer: {answer}")
    print(f" Response time: {round(duration, 2)}s")
    return answer

# Test with real questions!
ask_langgraph_agent("What is the total revenue from all sales?")
ask_langgraph_agent("Which customer spent the most money?")
ask_langgraph_agent("Was the last ingestion successful?")

/home/spark-1c5400ba-e438-4c98-908a-19/.ipykernel/15975/command-8476672931950070-570318607:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


LangGraph + OpenAI Agent created!

 Question: What is the total revenue from all sales?
 Answer: The total revenue from all sales is $4250.
 Response time: 11.04s

 Question: Which customer spent the most money?
 Answer: The customer who spent the most money is John with a total spending of $1200.
 Response time: 1.77s

 Question: Was the last ingestion successful?
 Answer: The last ingestion was successful. The source was "customer_api" and it ingested 5 rows at 2026-04-19 17:38:24.
 Response time: 1.64s


'The last ingestion was successful. The source was "customer_api" and it ingested 5 rows at 2026-04-19 17:38:24.'

[Trace(trace_id=tr-cab868929c0a60a463b2064a3059a467), Trace(trace_id=tr-8a4d85f88bc990eb05f6a45a10ab9060), Trace(trace_id=tr-3b51fd9b720aa899673626691067ecc0)]